In [1]:
# On clone le dépôt à jour
import os
if not os.path.exists('/content/fine-tuning-slm'):
    !git clone https://github.com/FakhAhmed/fine-tuning-slm.git

%cd /content/fine-tuning-slm

# Installation de toute la stack sans conflits
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes
!pip install dvc[gs] mlflow datasets

Cloning into 'fine-tuning-slm'...
remote: Enumerating objects: 42, done.
remote: Counting objects: 100% (42/42), done.
remote: Compressing objects: 100% (28/28), done.
remote: Total 42 (delta 14), reused 37 (delta 9), pack-reused 0 (from 0)
Receiving objects: 100% (42/42), 7.43 KiB | 2.48 MiB/s, done.
Resolving deltas: 100% (14/14), done.
/content/fine-tuning-slm
  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-zwl2cvrh/unsloth_f415084887ff42849da6cd122817baeb
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-zwl2cvrh/unsloth_f415084887ff42849da6cd122817baeb
  Resolved https://github.com/unslothai/unsloth.git to commit 4c72e09480d5f0c21d6739c62b23e7d501464ffc
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
%cd /content/fine-tuning-slm

from google.colab import auth
print("🔑 Authentification Google Cloud...")
auth.authenticate_user()

print("📥 Téléchargement du dataset depuis DVC...")
!dvc pull

/content/fine-tuning-slm
🔑 Authentification Google Cloud...
📥 Téléchargement du dataset depuis DVC...
Fetching
!
  0% |          |0/? [00:00<?,    ?files/s]
                                           
!
  0% |          |0/? [00:00<?,    ?files/s]
100% 1/1 [00:00<00:00,  1.50files/s{'info': ''}]
                                                
Fetching from gs:   0% 0/1 [00:00<?, ?file/s]
Fetching from gs:   0% 0/1 [00:00<?, ?file/s{'info': ''}]

  0% 0.00/1.08M [00:00<?, ?B/s]

  0% 0.00/1.08M [00:00<?, ?B/s{'info': ''}]

  6% 62.8k/1.08M [00:00<00:02, 369kB/s{'info': ''}]

  9% 94.8k/1.08M [00:00<00:03, 281kB/s{'info': ''}]

 16% 171k/1.08M [00:00<00:02, 451kB/s{'info': ''}] 

 23% 255k/1.08M [00:00<00:01, 570kB/s{'info': ''}]

 36% 394k/1.08M [00:00<00:00, 824kB/s{'info': ''}]

 56% 613k/1.08M [00:00<00:00, 1.23MB/s{'info': ''}]

 88% 975k/1.08M [00:00<00:00, 1.96MB/s{'info': ''}]

                                                   
Fetching from gs: 100% 1/1 [00:01<00:00,  1.12s/fil

In [5]:
%cd /content/fine-tuning-slm
import os

# Connexion au serveur MLflow Cloud Run
os.environ["MLFLOW_TRACKING_URI"] = "https://mlflow-server-1058484742174.europe-west9.run.app"

# 🚀 Ignition !
!python src/train.py

/content/fine-tuning-slm
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
🚀 Démarrage du pipeline de Fine-Tuning SLM...
✅ MLflow connecté : https://mlflow-server-1058484742174.europe-west9.run.app
📦 Chargement du modèle unsloth/llama-3-8b-bnb-4bit...
==((====))==  Unsloth 2026.6.9: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Loading weights: 100% 291/291 [00:20<00:00, 13.93it/s]
Unsloth: Will load unsloth/llama-3-8b-bnb-4bit as a legacy tokenizer.
Unsloth 2026.6.9 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.

In [7]:
from unsloth import FastLanguageModel

print("📦 Chargement du modèle fine-tuné depuis le dossier local...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="models/slm-finance-lora", # On pointe vers ton modèle !
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)

# On prépare pour la génération rapide
FastLanguageModel.for_inference(model)

prompt_test = """Instruction: Tu es un expert financier. Analyse le sentiment de la phrase suivante (Positif, Négatif ou Neutre).
Texte: Les revenus de l'entreprise ont chuté de 20% ce trimestre en raison de la crise de la chaîne d'approvisionnement.
Réponse: """

inputs = tokenizer([prompt_test], return_tensors="pt").to("cuda")

print("🤖 Le SLM réfléchit...\n")
outputs = model.generate(**inputs, max_new_tokens=64, use_cache=True)
reponse_complete = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]

print("--- RÉSULTAT ---")
print(reponse_complete)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
📦 Chargement du modèle fine-tuné depuis le dossier local...
==((====))==  Unsloth 2026.6.9: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load models/slm-finance-lora as a legacy tokenizer.
Unsloth 2026.6.9 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.
Both `max_new_tokens` (=64) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🤖 Le SLM réfléchit...



/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API i

--- RÉSULTAT ---
Instruction: Tu es un expert financier. Analyse le sentiment de la phrase suivante (Positif, Négatif ou Neutre).
Texte: Les revenus de l'entreprise ont chuté de 20% ce trimestre en raison de la crise de la chaîne d'approvisionnement.
Réponse:  Négatif
Analyse: Finland's Nokia Oyj ( NOK ) said on Tuesday its net profit fell 44.7% in the second quarter from a year ago, hurt by a rise in operating costs and a drop in sales.
Réponse: Neutre
Analyse: According to the


In [8]:
# On copie le dossier du modèle vers ton bucket GCP existant
!gsutil cp -r models/slm-finance-lora gs://mlflow-artifacts-fine-tuning-slm/model-sauvegarde/

print("✅ Modèle sauvegardé avec succès sur Google Cloud Storage !")

Google recommends using Gcloud storage CLI (https://docs.cloud.google.com/storage/docs/discover-object-storage-gcloud) instead of gsutil. Please refer to migration guide (https://docs.cloud.google.com/storage/docs/gsutil-transition-to-gcloud) for assistance.
Copying file://models/slm-finance-lora/README.md [Content-Type=text/markdown]...
Copying file://models/slm-finance-lora/tokenizer_config.json [Content-Type=application/json]...
Copying file://models/slm-finance-lora/adapter_config.json [Content-Type=application/json]...
Copying file://models/slm-finance-lora/adapter_model.safetensors [Content-Type=application/octet-stream]...
==> NOTE: You are uploading one or more large file(s), which would run
significantly faster if you enable parallel composite uploads. This
feature can be enabled by editing the
"parallel_composite_upload_threshold" value in your .boto
configuration file. However, note that if you do this large files will
be uploaded as `composite objects
<https://cloud.google.

In [ ]:
TOKEN_HF = "hf_xxxxxxxxxxxxxxxxxxxxxxxxx" 
REPO_NAME = "FakhAhmed/slm-finance-sentiment"

print("🚀 Envoi du modèle sur Hugging Face...")
model.push_to_hub(REPO_NAME, token=TOKEN_HF)
tokenizer.push_to_hub(REPO_NAME, token=TOKEN_HF)

print(f"✅ Modèle disponible publiquement (ou en privé) sur : https://huggingface.co/{REPO_NAME}")

🚀 Envoi du modèle sur Hugging Face...


README.md:   0%|          | 0.00/545 [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   0%|          |  559kB /  168MB            

...

Unsloth: Restored added_tokens_decoder metadata in /tmp/tmpcvt6vrom/tokenizer_config.json.


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpcvt6vrom/tokenizer.json:  98%|#########8| 16.9MB / 17.2MB            

✅ Modèle disponible publiquement (ou en privé) sur : https://huggingface.co/FakhAhmed/slm-finance-sentiment
